# Saudi Crime Cleaning Pipeline
This notebook loads raw Saudi crime data, cleans it step-by-step, and writes a cleaned CSV output.
Ensure the raw CSV is available in `data/raw/saudi_crime_raw.csv` and the output will be saved to `data/clean/saudi_crime_clean.csv`.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# Determine project root so the notebook works from either the workspace root or the notebooks folder
cwd = Path.cwd()
if (cwd / 'data' / 'raw' / 'saudi_crime_raw.csv').exists():
    project_root = cwd
elif (cwd.parent / 'data' / 'raw' / 'saudi_crime_raw.csv').exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError('Could not find data/raw/saudi_crime_raw.csv from the current working directory: ' + str(cwd))

raw_csv = project_root / 'data' / 'raw' / 'saudi_crime_raw.csv'
clean_csv = project_root / 'data' / 'clean' / 'saudi_crime_clean.csv'
clean_csv.parent.mkdir(parents=True, exist_ok=True)
print('Project root:', project_root)
print('Raw file:', raw_csv)
print('Clean output:', clean_csv)

Project root: /Users/sibtainahmedqureshi/SA-Crime-Analysis-2026
Raw file: /Users/sibtainahmedqureshi/SA-Crime-Analysis-2026/data/raw/saudi_crime_raw.csv
Clean output: /Users/sibtainahmedqureshi/SA-Crime-Analysis-2026/data/clean/saudi_crime_clean.csv


In [3]:
df = pd.read_csv(raw_csv)

print('=' * 55)
print('  SAUDI CRIME - DATA CLEANING PIPELINE')
print('=' * 55)
print(f'\nRAW SHAPE  : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nNULL COUNTS (before cleaning):')
print(df.isnull().sum()[df.isnull().sum() > 0].to_string())
print(f'\nDuplicate Case IDs : {df.duplicated(subset=["Case_ID"]).sum():,}')

  SAUDI CRIME - DATA CLEANING PIPELINE

RAW SHAPE  : 50,000 rows × 11 columns

NULL COUNTS (before cleaning):
Arrest_Date          877
Year_of_Arrest      1495
Month_of_Arrest     1495
Nationality         2524
Age_at_Arrest        447
Charge_3_Statute     532

Duplicate Case IDs : 593


In [5]:
# Step 1 — Drop fully duplicate rows
before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'[Step 1] Drop full duplicates          : -{before - len(df):,} rows')

[Step 1] Drop full duplicates          : -0 rows


In [6]:
# Step 2 — Parse & fix Arrest_Date
INVALID_VALS = {'N/A', '0000-00-00', '', 'TBD', 'nan', 'None'}

def parse_date(val):
    if pd.isnull(val) or str(val).strip() in INVALID_VALS:
        return pd.NaT
    try:
        return pd.to_datetime(val)
    except Exception:
        return pd.NaT

df['Arrest_Date'] = df['Arrest_Date'].apply(parse_date)
bad_dates = df['Arrest_Date'].isna().sum()
print(f'[Step 2] Unparseable dates → NaT       : {bad_dates:,} rows affected')

[Step 2] Unparseable dates → NaT       : 1,495 rows affected


In [7]:
# Step 3 — Drop rows with no valid date
df.dropna(subset=['Arrest_Date'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'[Step 3] Dropped rows (invalid date)   : -{bad_dates:,} rows')

[Step 3] Dropped rows (invalid date)   : -1,495 rows


In [8]:
# Step 4 — Re-derive Year & Month from clean date
df['Year_of_Arrest'] = df['Arrest_Date'].dt.year.astype('Int64')
df['Month_of_Arrest'] = df['Arrest_Date'].dt.strftime('%B')
print('[Step 4] Year & Month re-derived from Arrest_Date')

[Step 4] Year & Month re-derived from Arrest_Date


In [9]:
# Step 5 — Fix Age outliers → median imputation
df['Age_at_Arrest'] = pd.to_numeric(df['Age_at_Arrest'], errors='coerce')
outlier_mask = ~df['Age_at_Arrest'].between(14, 80)
n_outliers = outlier_mask.sum()
df.loc[outlier_mask, 'Age_at_Arrest'] = np.nan
median_age = df['Age_at_Arrest'].median()
df['Age_at_Arrest'] = df['Age_at_Arrest'].fillna(median_age).round(0).astype('Int64')
print(f'[Step 5] Age outliers → median ({median_age:.0f})     : {n_outliers:,} replaced')

[Step 5] Age outliers → median (32)     : 2,934 replaced


In [10]:
# Step 6 — Fill missing Nationality
n = df['Nationality'].isna().sum()
df['Nationality'] = df['Nationality'].fillna('Unknown / Refused')
print(f'[Step 6] Missing Nationality filled    : {n:,} rows')

[Step 6] Missing Nationality filled    : 2,451 rows


In [11]:
# Step 7 — Fill missing Charge_3_Statute
n = df['Charge_3_Statute'].isna().sum()
df['Charge_3_Statute'] = df['Charge_3_Statute'].fillna('Not Recorded')
print(f'[Step 7] Missing Charge_3 filled       : {n:,} rows')

[Step 7] Missing Charge_3 filled       : 515 rows


In [12]:
# Step 8 — Standardise all text columns
text_cols = [
    'Charge_1_Statute', 'Charge_2_Statute',
    'Charge_3_Statute', 'Charge_4_Statute',
    'Nationality', 'Region'
]
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()
print('[Step 8] Text columns stripped & standardised')

# Step 9 — Add Age Group column
bins = [13, 17, 24, 34, 44, 54, 64, 80]
labels = ['14-17', '18-24', '25-34', '35-44', '45-54', '55-64', '65+']
df['Age_Group'] = pd.cut(
    df['Age_at_Arrest'].astype(float),
    bins=bins, labels=labels, right=True
).astype(str)
print('[Step 9] Age_Group column added')

# Step 10 — Add Quarter column & sort
df['Quarter'] = 'Q' + df['Arrest_Date'].dt.quarter.astype(str)
df.sort_values('Arrest_Date', inplace=True)
df.reset_index(drop=True, inplace=True)
print('[Step 10] Quarter column added & data sorted by date')

[Step 8] Text columns stripped & standardised
[Step 9] Age_Group column added
[Step 10] Quarter column added & data sorted by date


In [13]:
print('\n' + '=' * 55)
print('  CLEAN DATA SUMMARY')
print('=' * 55)
print(f'Final shape  : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Remaining NULLs : {df.isnull().sum().sum()}')
print(f"Date range   : {df['Arrest_Date'].min().date()} → {df['Arrest_Date'].max().date()}")

df.to_csv(clean_csv, index=False)
print('\n✔ ' + str(clean_csv.name) + ' saved successfully')
print('=' * 55)


  CLEAN DATA SUMMARY
Final shape  : 48,505 rows × 13 columns
Remaining NULLs : 0
Date range   : 2014-01-01 → 2023-12-28

✔ saudi_crime_clean.csv saved successfully
